# 02 · O que o pré-processamento fez com os dados

O arquivo entregue tem 253.680 respondentes. O **BRFSS 2015 original do CDC tem
441.456**. Este notebook reconstrói as 22 colunas a partir da fonte e mede o viés.

> Documento completo: [`docs/05-comparacao-brfss-original.md`](../docs/05-comparacao-brfss-original.md)


In [1]:
import sys, json
from pathlib import Path

# a raiz e onde existe src/ — funciona rodando de notebooks/ ou da raiz do repo
RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np, pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

GOLD = RAIZ / "data" / "processed" / "gold"
def ler(nome, base=GOLD):
    return json.loads((base / nome).read_text(encoding="utf-8"))


## A prova de integridade


In [2]:
from diabetes.schema import ESQUEMA
EXT = RAIZ / "data" / "external" / "brfss2015"

prof = pd.read_parquet(RAIZ / "data/processed/diabetes_silver.parquet")[list(ESQUEMA)]
rec  = pd.read_parquet(EXT / "brfss2015_reconstruido.parquet")[list(ESQUEMA)]

iguais = (prof.reset_index(drop=True).values == rec.reset_index(drop=True).values)
print(f"células idênticas   : {iguais.mean()*100:.6f}%")
print(f"linhas 100% iguais  : {iguais.all(axis=1).sum():,} de {len(prof):,}")


células idênticas   : 100.000000%
linhas 100% iguais  : 253,680 de 253,680


**Identidade perfeita.** Isso prova quatro coisas de uma vez: a extração do PDF
está correta, as regras de derivação são as que geraram o arquivo, o download
está íntegro, e a ordem das linhas foi preservada.


## A cascata de exclusões — quem foi descartado


In [3]:
casc = pd.DataFrame(ler("_cascata_exclusoes.json", EXT)["cascata"])
casc = casc[(casc.etapa != "final") & (casc.excluidos > 0)].sort_values("excluidos", ascending=False)
casc["% das exclusões"] = (casc.excluidos / casc.excluidos.sum() * 100).round(1)
casc[["regra", "variavel", "criterio", "excluidos", "% das exclusões"]].head(8)


,regra,variavel,criterio,excluidos,% das exclusões
0,—,(qualquer),valor ausente,97850,52.1
22,renda_faixa,INCOME2,"descarta [77, 99]",34251,18.2
9,atividade_fisica,_TOTINDA,descarta [9],15397,8.2
10,frutas,_FRTLT1,descarta [9],7636,4.1
11,vegetais,_VEGLT1,descarta [9],7608,4.1
4,exame_colesterol,_CHOLCHK,descarta [9],4342,2.3
17,saude_fisica_dias,PHYSHLTH,"descarta [77, 99]",3629,1.9
12,alcool_excessivo,_RFDRHV5,descarta [9],3523,1.9


## O viés que isso produz


In [4]:
vies = ler("_analise_vies.json", EXT)
pd.DataFrame(vies["prevalencia"])[
    ["estimativa", "n", "n_efetivo", "diabetes_%", "ic95_diabetes"]]


,estimativa,n,n_efetivo,diabetes_%,ic95_diabetes
0,"a · arquivo entregue, SEM peso",253680,253680,13.933,[13.80; 14.07]
1,"b · BRFSS completo, SEM peso",440658,440658,12.993,[12.89; 13.09]
2,"c · BRFSS completo, COM _LLCPWT",440658,109019,10.500,[10.32; 10.68]
3,"d · subamostra analitica, COM _LLCPWT",253680,64117,12.280,[12.03; 12.53]


In [5]:
a = 13.933   # arquivo entregue, sem peso
b = 12.993   # BRFSS completo, sem peso
c = 10.500   # BRFSS completo, com peso
print(f"descarte de 42,5% da amostra : {b-a:+.2f} p.p.  ({(a-b)/(a-c)*100:.0f}% do viés)")
print(f"peso amostral descartado     : {c-b:+.2f} p.p.  ({(b-c)/(a-c)*100:.0f}% do viés)")
print(f"\nviés total: {a-c:+.2f} p.p. — superestimação de {(a/c-1)*100:.1f}%")


descarte de 42,5% da amostra : -0.94 p.p.  (27% do viés)
peso amostral descartado     : -2.49 p.p.  (73% do viés)

viés total: +3.43 p.p. — superestimação de 32.7%


> **O resultado contraintuitivo:** a maior parte do viés **não** vem de terem
> jogado fora 187.776 pessoas. Vem de terem jogado fora **três colunas**
> (`_LLCPWT`, `_STSTR`, `_PSU`).


## O achado mais grave: o arquivo é uma amostra de quem tem acesso


In [6]:
pd.DataFrame([
    ["% fez exame de colesterol", 96.27, 77.93],
    ["% com plano de saúde",      95.11, 87.83],
    ["% sem consulta por custo",   8.42, 13.27],
], columns=["indicador", "arquivo entregue", "população (ponderada)"]).assign(
    viés_pp=lambda d: (d["arquivo entregue"] - d["população (ponderada)"]).round(2))


,indicador,arquivo entregue,população (ponderada),viés_pp
0,% fez exame de colesterol,96.27,77.93,18.34
1,% com plano de saúde,95.11,87.83,7.28
2,% sem consulta por custo,8.42,13.27,-4.85


Quase um em cada quatro americanos nunca fez exame de colesterol. No arquivo
entregue, é **um em vinte e sete**.

Consequência que reescreve o plano de análise: `exame_colesterol` é quase
constante e não representa a população; qualquer conclusão sobre **desigualdade
de acesso** feita neste arquivo está estruturalmente comprometida.
